In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive



QCDP-BiFormer — MODULE 9: MULTI-LABEL CLASSIFICATION


Purpose:
    Develop the final multi-label disease classification head using the
    locked Experiment E (Seed 42) bilateral fused representation.

Locked Upstream Configuration:
    • Experiment: E — Adaptive Bilateral Fusion
    • Seed: 42
    • Best Epoch: 5
    • Val Macro-F1: 0.561936
    • Val Micro-F1: 0.535966
    • Exact Match: 0.175799
    • Locked artifact: stage2_experiment_e_locked_seed42.pt

Strict Constraints:
    • Experiment E is FROZEN.
    • Cross-Eye representations are FROZEN.
    • Adaptive bilateral fusion is FROZEN.
    • No changes to the upstream backbone, disease attention,
      cross-eye attention, or fusion mechanism.
    • Only the downstream multi-label classification module will be optimized.

Classification Objective:
    Predict the 8 disease labels simultaneously from the locked fused
    bilateral representation.

Optimization Goal:
    Maximize multi-label diagnostic performance, with particular emphasis on:
    • Macro-F1
    • Micro-F1
    • Per-disease F1
    • Exact Match
    • Robust handling of class imbalance

Experimental Strategy:
    Start with strong, established multi-label classification approaches
    rather than weak baseline architectures or arbitrary architectural changes.

    Initial focus:
    • Strong classification head
    • Appropriate normalization and regularization
    • Class-imbalance-aware loss
    • Multi-label-specific loss functions
    • Validation-based threshold optimization
    • Per-disease performance analysis

Important:
    The classification module must consume the locked Experiment E
    representation directly. Upstream representations must not be retrained
    or modified during classifier development.



In [3]:
# =============================================================================
# QCDP-BiFormer — MODULE 9
# MULTI-LABEL CLASSIFICATION
# CELL 1 — ENVIRONMENT + LOCKED RESOURCES
# =============================================================================

import os
import random
import json
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from pathlib import Path

# -----------------------------------------------------------------------------
# 1. GOOGLE DRIVE
# -----------------------------------------------------------------------------

from google.colab import drive

drive.mount('/content/drive')

ROOT = Path("/content/drive/My Drive/Eye Disease/Dataset")

assert ROOT.exists(), f"Dataset root not found: {ROOT}"

print("=" * 80)
print("QCDP-BiFormer — MULTI-LABEL CLASSIFICATION")
print("=" * 80)
print(f"Dataset root: {ROOT}")
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

# -----------------------------------------------------------------------------
# 2. REPRODUCIBILITY
# -----------------------------------------------------------------------------

SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

# Deterministic behavior for evaluation/reproducibility
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

print(f"\nGlobal seed: {SEED}")

# -----------------------------------------------------------------------------
# 3. LOCKED EXPERIMENT
# -----------------------------------------------------------------------------

LOCKED_EXPERIMENT = "experiment_e"
LOCKED_SEED = 42

LOCKED_ARTIFACT = ROOT / "stage2_experiment_e_locked_seed42.pt"

assert LOCKED_ARTIFACT.exists(), (
    f"Locked Experiment E artifact not found:\n{LOCKED_ARTIFACT}"
)

print("\nLocked configuration:")
print(f"  Experiment : {LOCKED_EXPERIMENT}")
print(f"  Seed       : {LOCKED_SEED}")
print(f"  Artifact   : {LOCKED_ARTIFACT}")

# -----------------------------------------------------------------------------
# 4. DATASET / LABEL DEFINITIONS
# -----------------------------------------------------------------------------

TRAIN_DF_PATH = ROOT / "stage2_train_df.csv"
VAL_DF_PATH   = ROOT / "val_patient_df.csv"

assert TRAIN_DF_PATH.exists(), f"Missing: {TRAIN_DF_PATH}"
assert VAL_DF_PATH.exists(), f"Missing: {VAL_DF_PATH}"

train_df = pd.read_csv(TRAIN_DF_PATH)
val_df   = pd.read_csv(VAL_DF_PATH)

# Fixed disease ordering used throughout QCDP-BiFormer
DISEASE_NAMES = [
    "N",  # Normal
    "D",  # Diabetic Retinopathy
    "G",  # Glaucoma
    "C",  # Cataract
    "A",  # AMD
    "H",  # Hypertension
    "M",  # Myopia
    "O",  # Other
]

NUM_CLASSES = len(DISEASE_NAMES)

print("\nDataset:")
print(f"  Train samples : {len(train_df)}")
print(f"  Val samples   : {len(val_df)}")
print(f"  Classes       : {NUM_CLASSES}")
print(f"  Label order   : {DISEASE_NAMES}")

# -----------------------------------------------------------------------------
# 5. EXISTING LOCKED / UPSTREAM RESOURCES
# -----------------------------------------------------------------------------

RESOURCE_FILES = {
    "stage1_train_df": ROOT / "stage1_train_df.csv",
    "stage2_train_df": ROOT / "stage2_train_df.csv",
    "stage1_quality_scores": ROOT / "stage1_quality_scores.csv",

    "disease_aware_features": ROOT / "disease_aware_features.pt",
    "disease_attention_weights": ROOT / "disease_attention_weights.pt",
    "disease_prototypes": ROOT / "disease_prototypes.pt",

    "cross_eye_train": ROOT / "stage2_train_cross_eye_features_final.pt",
    "cross_eye_val": ROOT / "stage2_val_cross_eye_features_final.pt",

    "locked_experiment_e": LOCKED_ARTIFACT,
}

print("\nUpstream resources:")
for name, path in RESOURCE_FILES.items():
    status = "FOUND" if path.exists() else "NOT FOUND"
    print(f"  [{status:>9}] {name}: {path.name}")

# -----------------------------------------------------------------------------
# 6. LOAD LOCKED EXPERIMENT E ARTIFACT
# -----------------------------------------------------------------------------

print("\n" + "-" * 80)
print("Loading locked Experiment E artifact...")
print("-" * 80)

locked_artifact = torch.load(
    LOCKED_ARTIFACT,
    map_location="cpu",
    weights_only=False
)

print("Artifact loaded successfully.")
print(f"Artifact type: {type(locked_artifact)}")

# Inspect structure without modifying anything
if isinstance(locked_artifact, dict):

    print("\nArtifact contents:")
    for key, value in locked_artifact.items():

        if torch.is_tensor(value):
            print(
                f"  {key}: Tensor "
                f"shape={tuple(value.shape)}, "
                f"dtype={value.dtype}"
            )

        elif isinstance(value, dict):
            print(
                f"  {key}: dict "
                f"({len(value)} entries)"
            )

        elif isinstance(value, (list, tuple)):
            print(
                f"  {key}: {type(value).__name__} "
                f"({len(value)} entries)"
            )

        else:
            print(
                f"  {key}: "
                f"{type(value).__name__} = {value}"
            )

# -----------------------------------------------------------------------------
# 7. LOCKED REFERENCE METRICS
# -----------------------------------------------------------------------------

LOCKED_REFERENCE = {
    "best_epoch": 5,
    "val_loss": 0.791240,
    "micro_f1": 0.535966,
    "macro_f1": 0.561936,
    "exact_match": 0.175799,
}

print("\n" + "-" * 80)
print("LOCKED EXPERIMENT E / SEED 42 REFERENCE")
print("-" * 80)

for metric, value in LOCKED_REFERENCE.items():
    print(f"{metric:>15}: {value}")

# -----------------------------------------------------------------------------
# 8. GLOBAL CLASSIFICATION CONFIGURATION
# -----------------------------------------------------------------------------

FEATURE_DIM = 768
NUM_CLASSES = 8

# We will NOT modify these upstream representations.
UPSTREAM_FROZEN = True

print("\n" + "=" * 80)
print("MODULE 9 INITIALIZATION COMPLETE")
print("=" * 80)

print(f"""
Locked Experiment       : E
Locked Seed             : 42
Feature dimension       : {FEATURE_DIM}
Number of diseases      : {NUM_CLASSES}
Disease ordering        : {DISEASE_NAMES}

Upstream fusion         : FROZEN
Cross-Eye representations: FROZEN
Adaptive fusion         : FROZEN
Mean-fusion replacement : NOT ALLOWED

Next step:
    Inspect the artifact structure and determine exactly which
    fused representation should enter the classification head.
""")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
QCDP-BiFormer — MULTI-LABEL CLASSIFICATION
Dataset root: /content/drive/My Drive/Eye Disease/Dataset
PyTorch version: 2.11.0+cpu
CUDA available: False

Global seed: 42

Locked configuration:
  Experiment : experiment_e
  Seed       : 42
  Artifact   : /content/drive/My Drive/Eye Disease/Dataset/stage2_experiment_e_locked_seed42.pt

Dataset:
  Train samples : 2141
  Val samples   : 504
  Classes       : 8
  Label order   : ['N', 'D', 'G', 'C', 'A', 'H', 'M', 'O']

Upstream resources:
  [    FOUND] stage1_train_df: stage1_train_df.csv
  [    FOUND] stage2_train_df: stage2_train_df.csv
  [    FOUND] stage1_quality_scores: stage1_quality_scores.csv
  [    FOUND] disease_aware_features: disease_aware_features.pt
  [    FOUND] disease_attention_weights: disease_attention_weights.pt
  [    FOUND] disease_prototypes: disease_prototypes.pt
  [    FOUND] cross_eye_trai

CELL 2 — LOCKED FUSION INSPECTION & FEATURE SOURCE VERIFICATION

Objective:
    Determine the exact representation produced by the locked Experiment E
    configuration that will serve as input to the multi-label classifier.

This cell performs inspection only. No model weights, upstream features, or
fusion parameters are modified.

Verification Targets:
    • Confirm the locked Experiment E architecture.
    • Inspect the exact adaptive fusion configuration.
    • Identify the dimensionality of the fused representation.
    • Inspect all available Stage 2 feature artifacts.
    • Determine whether F_fused is directly stored or must be reconstructed
      from the saved upstream representations and locked fusion weights.
    • Verify train/validation feature availability and dimensions.

Strict Lock:
    Experiment E — Seed 42 remains completely frozen.

    No changes are permitted to:
        • Backbone
        • Quality conditioning
        • Disease-aware attention
        • Cross-eye bilateral attention
        • Adaptive bilateral fusion

Important:
    The classifier will only be designed after the exact F_fused representation
    and its train/validation correspondence have been verified.



In [4]:
# =============================================================================
# QCDP-BiFormer — MODULE 9
# CELL 2 — LOCKED FUSION INSPECTION & FEATURE SOURCE VERIFICATION
# =============================================================================

print("=" * 80)
print("CELL 2 — LOCKED FUSION INSPECTION & FEATURE SOURCE VERIFICATION")
print("=" * 80)

# -----------------------------------------------------------------------------
# 1. INSPECT LOCKED ARCHITECTURE CONFIGURATION
# -----------------------------------------------------------------------------

print("\n" + "-" * 80)
print("1. LOCKED EXPERIMENT E ARCHITECTURE")
print("-" * 80)

architecture = locked_artifact.get("architecture", None)

if architecture is None:
    print("WARNING: No architecture dictionary found in artifact.")
else:
    print(f"Architecture entries: {len(architecture)}")

    for key, value in architecture.items():
        print(f"\n[{key}]")

        if isinstance(value, dict):
            for sub_key, sub_value in value.items():
                print(f"  {sub_key}: {sub_value}")

        elif isinstance(value, (list, tuple)):
            print(f"  {value}")

        else:
            print(f"  {value}")


# -----------------------------------------------------------------------------
# 2. INSPECT COMPLETE ARTIFACT METADATA
# -----------------------------------------------------------------------------

print("\n" + "-" * 80)
print("2. COMPLETE LOCKED ARTIFACT METADATA")
print("-" * 80)

for key, value in locked_artifact.items():

    if key == "architecture":
        continue

    if isinstance(value, dict):
        print(f"\n{key}:")
        for sub_key, sub_value in value.items():
            print(f"  {sub_key}: {sub_value}")

    elif torch.is_tensor(value):
        print(
            f"{key}: Tensor "
            f"shape={tuple(value.shape)}, "
            f"dtype={value.dtype}"
        )

    else:
        print(f"{key}: {value}")


# -----------------------------------------------------------------------------
# 3. INSPECT ALL STAGE-2 PT FILES
# -----------------------------------------------------------------------------

print("\n" + "-" * 80)
print("3. AVAILABLE STAGE-2 FEATURE ARTIFACTS")
print("-" * 80)

stage2_pt_files = sorted(ROOT.glob("stage2*.pt"))

if not stage2_pt_files:
    print("No stage2*.pt files found.")

else:
    for path in stage2_pt_files:

        print(f"\nFILE: {path.name}")
        print(f"SIZE: {path.stat().st_size / (1024**2):.2f} MB")

        try:
            obj = torch.load(
                path,
                map_location="cpu",
                weights_only=False
            )

            print(f"TYPE: {type(obj)}")

            if isinstance(obj, dict):

                print("CONTENTS:")

                for key, value in obj.items():

                    if torch.is_tensor(value):
                        print(
                            f"  {key}: Tensor "
                            f"shape={tuple(value.shape)}, "
                            f"dtype={value.dtype}"
                        )

                    elif isinstance(value, dict):
                        print(
                            f"  {key}: dict "
                            f"({len(value)} entries)"
                        )

                    elif isinstance(value, (list, tuple)):
                        print(
                            f"  {key}: {type(value).__name__} "
                            f"({len(value)} entries)"
                        )

                    else:
                        print(
                            f"  {key}: "
                            f"{type(value).__name__}"
                        )

            elif torch.is_tensor(obj):
                print(
                    f"Tensor shape={tuple(obj.shape)}, "
                    f"dtype={obj.dtype}"
                )

        except Exception as e:
            print(f"  ERROR LOADING FILE: {e}")


# -----------------------------------------------------------------------------
# 4. INSPECT KNOWN UPSTREAM FEATURE FILES
# -----------------------------------------------------------------------------

print("\n" + "-" * 80)
print("4. KNOWN UPSTREAM FEATURE FILES")
print("-" * 80)

known_feature_files = [
    ROOT / "disease_aware_features.pt",
    ROOT / "disease_attention_weights.pt",
    ROOT / "disease_prototypes.pt",
    ROOT / "stage2_train_cross_eye_features_final.pt",
    ROOT / "stage2_val_cross_eye_features_final.pt",
]

for path in known_feature_files:

    print(f"\n{'=' * 60}")
    print(f"FILE: {path.name}")
    print(f"EXISTS: {path.exists()}")

    if not path.exists():
        continue

    try:
        obj = torch.load(
            path,
            map_location="cpu",
            weights_only=False
        )

        print(f"TYPE: {type(obj)}")

        if torch.is_tensor(obj):

            print(f"SHAPE: {tuple(obj.shape)}")
            print(f"DTYPE: {obj.dtype}")

        elif isinstance(obj, dict):

            print(f"NUMBER OF KEYS: {len(obj)}")

            for key, value in obj.items():

                if torch.is_tensor(value):
                    print(
                        f"  {key}: Tensor "
                        f"shape={tuple(value.shape)}, "
                        f"dtype={value.dtype}"
                    )

                elif isinstance(value, dict):
                    print(
                        f"  {key}: dict "
                        f"({len(value)} entries)"
                    )

                elif isinstance(value, (list, tuple)):
                    print(
                        f"  {key}: {type(value).__name__} "
                        f"({len(value)} entries)"
                    )

                else:
                    print(
                        f"  {key}: "
                        f"{type(value).__name__} = {value}"
                    )

        else:
            print(f"OBJECT: {obj}")

    except Exception as e:
        print(f"ERROR: {e}")


# -----------------------------------------------------------------------------
# 5. SEARCH FOR POTENTIAL FUSED-FEATURE FILES
# -----------------------------------------------------------------------------

print("\n" + "-" * 80)
print("5. SEARCHING FOR POTENTIAL FUSED REPRESENTATIONS")
print("-" * 80)

all_pt_files = sorted(ROOT.glob("*.pt"))

fusion_keywords = [
    "fused",
    "fusion",
    "bilateral",
    "experiment_e",
    "adaptive"
]

candidate_files = []

for path in all_pt_files:

    name_lower = path.name.lower()

    if any(keyword in name_lower for keyword in fusion_keywords):
        candidate_files.append(path)

if candidate_files:

    print("Potentially relevant files:")

    for path in candidate_files:
        print(
            f"  • {path.name} "
            f"({path.stat().st_size / (1024**2):.2f} MB)"
        )

else:
    print("No additional fusion-specific .pt files found.")


# -----------------------------------------------------------------------------
# 6. PRELIMINARY FEATURE DIMENSION CHECK
# -----------------------------------------------------------------------------

print("\n" + "-" * 80)
print("6. PRELIMINARY FEATURE DIMENSION CHECK")
print("-" * 80)

tensor_candidates = []

for path in all_pt_files:

    try:
        obj = torch.load(
            path,
            map_location="cpu",
            weights_only=False
        )

        if torch.is_tensor(obj):
            tensor_candidates.append(
                (path.name, tuple(obj.shape))
            )

        elif isinstance(obj, dict):

            for key, value in obj.items():

                if torch.is_tensor(value):
                    tensor_candidates.append(
                        (
                            f"{path.name} -> {key}",
                            tuple(value.shape)
                        )
                    )

    except Exception:
        pass

if tensor_candidates:

    for name, shape in tensor_candidates:
        print(f"  {name}: {shape}")

else:
    print("No directly accessible tensor candidates found.")


# -----------------------------------------------------------------------------
# 7. FINAL STATUS
# -----------------------------------------------------------------------------

print("\n" + "=" * 80)
print("CELL 2 COMPLETE")
print("=" * 80)

print("""
No upstream parameters were modified.

The output above is being used to determine:
    1. Exact Experiment E architecture
    2. Exact fusion representation
    3. Feature dimensionality
    4. Available train/validation feature tensors
    5. Whether F_fused is already saved or must be reconstructed

DO NOT build or train the classification head yet.
""")

CELL 2 — LOCKED FUSION INSPECTION & FEATURE SOURCE VERIFICATION

--------------------------------------------------------------------------------
1. LOCKED EXPERIMENT E ARCHITECTURE
--------------------------------------------------------------------------------
Architecture entries: 11

[gate_input_dim]
  18

[hidden_dim]
  16

[num_classes]
  8

[temperature]
  1.0

[residual_mean_anchor]
  False

[cross_eye_frozen]
  True

[class_wise_gating]
  True

[quality_information]
  True

[disease_relevance]
  True

[sigmoid_gating]
  True

[class_representation_dim]
  768

--------------------------------------------------------------------------------
2. COMPLETE LOCKED ARTIFACT METADATA
--------------------------------------------------------------------------------
experiment: Experiment E — Strict Class-wise Adaptive Fusion
experiment_key: experiment_e
seed: 42
best_epoch: 5
best_val_loss: 0.79124
val_micro_f1: 0.535966
val_macro_f1: 0.561936
exact_match: 0.175799
status: LOCKED

------